# Quantization Analysis — Results Visualization

This notebook loads experiment results produced by `run_sweep.py` and generates
tables and plots comparing accuracy across models and bit-widths.

In [ ]:
import json
import glob
import os

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "results")
if not os.path.isdir(RESULTS_DIR):
    RESULTS_DIR = "results"

print(f"Loading results from: {RESULTS_DIR}")

## 1. Load Results

In [ ]:
records = []
for path in sorted(glob.glob(os.path.join(RESULTS_DIR, "*.json"))):
    with open(path) as f:
        records.append(json.load(f))

if not records:
    raise FileNotFoundError(
        f"No result JSON files found in {RESULTS_DIR}. "
        "Run `python run_sweep.py` first."
    )

df = pd.DataFrame(records)
print(f"Loaded {len(df)} experiment results across {df['model'].nunique()} models.")
df.head()

## 2. Summary Table — Baseline vs. PTQ vs. QAT

In [ ]:
summary = df.pivot_table(
    index="model",
    columns="bit_width",
    values=["baseline_accuracy", "ptq_accuracy", "qat_accuracy"],
    aggfunc="first",
)

# Flatten the multi-level column index for readability
summary.columns = [f"{metric} (w{bw})" for metric, bw in summary.columns]
summary

## 3. PTQ Accuracy vs. Bit-width (Line Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for model_name, grp in df.groupby("model"):
    grp_sorted = grp.sort_values("bit_width")
    ax.plot(
        grp_sorted["bit_width"],
        grp_sorted["ptq_accuracy"],
        marker="o",
        linewidth=2,
        label=model_name,
    )
    # Dashed baseline
    baseline = grp_sorted["baseline_accuracy"].iloc[0]
    if baseline is not None:
        ax.axhline(baseline, linestyle="--", alpha=0.3)

ax.set_xlabel("Bit-width", fontsize=13)
ax.set_ylabel("PTQ Accuracy", fontsize=13)
ax.set_title("Post-Training Quantization: Accuracy vs. Bit-width", fontsize=14)
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(sorted(df["bit_width"].unique()))
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "ptq_accuracy_vs_bitwidth.png"), dpi=150)
plt.show()

## 4. Accuracy Drop from Baseline (Bar Chart)

In [ ]:
df["ptq_drop"] = df["baseline_accuracy"] - df["ptq_accuracy"]

fig, ax = plt.subplots(figsize=(12, 6))

models = sorted(df["model"].unique())
bit_widths = sorted(df["bit_width"].unique())
n_models = len(models)
bar_width = 0.8 / len(bit_widths)

for i, bw in enumerate(bit_widths):
    subset = df[df["bit_width"] == bw].set_index("model").reindex(models)
    positions = [j + i * bar_width for j in range(n_models)]
    ax.bar(
        positions,
        subset["ptq_drop"],
        width=bar_width,
        label=f"{bw}-bit",
        alpha=0.85,
    )

ax.set_xticks([j + bar_width * (len(bit_widths) - 1) / 2 for j in range(n_models)])
ax.set_xticklabels(models, rotation=30, ha="right", fontsize=10)
ax.set_ylabel("Accuracy Drop (Baseline - PTQ)", fontsize=13)
ax.set_title("Accuracy Drop by Model and Bit-width", fontsize=14)
ax.legend(title="Bit-width", fontsize=10)
ax.axhline(0, color="black", linewidth=0.5)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "accuracy_drop_bar.png"), dpi=150)
plt.show()

## 5. QAT Recovery (if available)

In [ ]:
df_qat = df.dropna(subset=["qat_accuracy"])

if df_qat.empty:
    print("No QAT results found. Re-run sweep with --run-qat to generate QAT data.")
else:
    fig, ax = plt.subplots(figsize=(10, 6))

    for model_name, grp in df_qat.groupby("model"):
        grp_sorted = grp.sort_values("bit_width")
        ax.plot(
            grp_sorted["bit_width"],
            grp_sorted["ptq_accuracy"],
            marker="s",
            linestyle="--",
            alpha=0.5,
            label=f"{model_name} (PTQ)",
        )
        ax.plot(
            grp_sorted["bit_width"],
            grp_sorted["qat_accuracy"],
            marker="o",
            linewidth=2,
            label=f"{model_name} (QAT)",
        )

    ax.set_xlabel("Bit-width", fontsize=13)
    ax.set_ylabel("Accuracy", fontsize=13)
    ax.set_title("QAT Recovery: PTQ vs. QAT Accuracy", fontsize=14)
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(sorted(df_qat["bit_width"].unique()))
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "qat_recovery.png"), dpi=150)
    plt.show()

## 6. Per-Model Detail Cards

In [ ]:
for model_name, grp in df.groupby("model"):
    grp_sorted = grp.sort_values("bit_width")
    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"  Dataset: {grp_sorted['dataset'].iloc[0]}")
    print(f"  Quantized layers: {grp_sorted['layer_types'].iloc[0]}")
    print(f"{'='*50}")

    card = grp_sorted[[
        "bit_width", "baseline_accuracy", "ptq_accuracy", "qat_accuracy"
    ]].copy()
    card["ptq_drop"] = card["baseline_accuracy"] - card["ptq_accuracy"]
    card = card.rename(columns={
        "bit_width": "Bits",
        "baseline_accuracy": "Baseline",
        "ptq_accuracy": "PTQ",
        "qat_accuracy": "QAT",
        "ptq_drop": "PTQ Drop",
    })
    display(card.set_index("Bits"))